# Create PS Features

`2025-05-25`  
Una vez que ya tengo todos los datos necesarios es momento de:
1. Crear las unidades de tratamiento (hacer el grid)
2. Asignar los features a cada uno de los grids


`2025-08-06`
#### Features
- Speed Camera (bool): wether or not a given cell in the grid has a speed camera in it
- Número de vialidades (int)
- Máximo de carriles (int)
- Direcciones (bool)
- Número de vialidades primarias (velocidad máxima 50 km/h)
- Número de vías de acceso controlado (velocidad máxima 80 km/h)
- Nivel de la vialidad (`zorder`)
- Afluencia promedio mensual previa a la implementación del programa en la estación de metro más cercana
  - *Sugerencia:* estandarizar con min-max para que los valores estén entre 0 y 1
- Distancia a la estación de metro más cercana (0 si tiene una estación) (utilizando `metro-station-coordinates`)
- Número total de incidentes viales
- Número de incidentes menores, `pics` y `fcs`

`next steps`  
- Calcular el PS
- Find best matches
- Actually do the fucking study

`2025-08-07`  
Bien Mariano, al parecer sí puedes trabajar dos días seguidos en tu tesis y no te vas a morir.

In [1]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import LineString, MultiLineString, Polygon
import warnings

warnings.filterwarnings("ignore")
PATH_DATA = '../data'

In [2]:
# Leemos los datos
# Ejes viales
vialidades = (
    gpd
    .read_file(os.path.join(PATH_DATA, 'vialidades.json'))
    .assign(
        tipo_vialidad=lambda x: x.TIPO_VIA.map({"Vía primaria":"primaria", 'Vía de acceso controlado':'acceso_controlado'}),
        carriles=lambda x: pd.to_numeric(x.CARRILES, errors='raise'),
        sentidos=lambda x: x.CIRCULA.map({"Un sentido":1, "Dos sentidos":2, "Un sentido con carril de contraflujo":2})
    )
    .rename({'NIVEL':'nivel', 'NOMENCLAT':'calle', 'NOMBRE':'vialidad', 'ID_VIA':'id_via'}, axis=1)
    [['id_via', 'vialidad', 'calle', 'tipo_vialidad', 'carriles', 'sentidos', 'nivel', 'geometry']]
)

# Speed Cameras
speed_cameras = gpd.read_file(
    os.path.join(
        PATH_DATA,
        'fotocivicas-ubicacion-puntos',
        'fotocivicas-ubicacion-puntos.shp'
    )
)

# Afluencia mensual y coordenadas
afluencia = pd.read_parquet(os.path.join(PATH_DATA, "afluencia-metro-mensual.parquet"))
coordinates = (
    pd
    .read_parquet(os.path.join(PATH_DATA, "metro-station-coordinates.parquet"))
    .pipe(lambda df: (
        gpd
        .GeoDataFrame(
            data=df.drop(['latitude', 'longitude'], axis=1),
            geometry=gpd.points_from_xy(df.longitude, df.latitude),
            crs="EPSG:4326"
        )
    ))
    .set_geometry('geometry')
)

# Incidentes viales
accidents = (
    pd
    .read_parquet(os.path.join(PATH_DATA, "classified-incidents.parquet"))
    .pipe(lambda df: (
        gpd
        .GeoDataFrame(
            data=df.drop(['latitude', 'longitude'], axis=1),
            geometry=gpd.points_from_xy(df.longitude, df.latitude),
            crs="EPSG:4326"
        )
    ))
)

### A. Building the Grid

In [3]:
def build_grid(df:gpd.GeoDataFrame, n:int) -> gpd.GeoDataFrame:
    '''
    Esta función recibe un dataframe con la información de geometría de líneas y regresa una cuadrícula 
    del tamaño n especificado, pero solo aquellas que tengan overlap con las líneas dadas.
    
    :param df: GeoDataFrame con LineStrings
    :param n: total de cuadrados a hacer
    
    :return: gpd.GeoDataFrame con dos columnas grid_id y geometry 
    '''
    def create_grid(upper_left, lower_right, n):
        minx, maxy = upper_left
        maxx, miny = lower_right
    
        # Calcular tamaño de cada celda
        cell_width = (maxx - minx) / n
        cell_height = (maxy - miny) / n
    
        # Crear una lista para guardar los polígonos
        polygons = []
        
        # Crear cuadrícula
        for i in range(n):
            for j in range(n):
                x1 = minx + i * cell_width
                y1 = maxy - j * cell_height
                x2 = x1 + cell_width
                y2 = y1 - cell_height
                polygons.append(Polygon([(x1, y1), (x2, y1), (x2, y2), (x1, y2)]))
        
        # Crear un GeoDataFrame a partir de los polígonos
        result = (
            gpd
            .GeoDataFrame({'geometry': polygons}, crs="EPSG:4326")
            .reset_index()
            .rename({'index':'grid_id'}, axis=1)
            .assign(centroid=lambda x: x.centroid) # para calcular la distancia a la estación del metro más cercana
        )
    
        return result
        
    minx, miny, maxx, maxy = df.total_bounds

    upper_left = (minx, maxy)
    lower_right = (maxx, miny)
    
    grid = create_grid(upper_left, lower_right, n)
    
    # Creamos el match para quitar los del grid que no tengan ninguno
    squares_with_streets = (
        gpd
        .sjoin(df, grid, how="left", predicate="intersects")
        [['grid_id']]
        .drop_duplicates()
    )
    
    final_grid = grid.merge(squares_with_streets, on='grid_id')

    # Calculamos el length en metros (para referencia)
    import math
    area = final_grid.to_crs(epsg=6933).area.values[0]
    length = round(math.sqrt(area), 2)

    return {
        "grid":final_grid,
        "length_m":length
    }

In [4]:
n = 500
grid_response = build_grid(vialidades, n)
grid = grid_response.get("grid")
length = grid_response.get("length_m")

In [5]:
length

74.08

### B. Assigning the features

In [6]:
# I. Speed Cameras
grid_has_camera = (
    gpd
    .sjoin(
        left_df=grid,
        right_df=speed_cameras,
        how="left", 
        predicate="intersects"
    )
    .assign(has_camera=lambda x: ~x.index_right.isna())
    .astype({'has_camera':'int'})
    [['grid_id', 'has_camera']]
    .drop_duplicates(keep='last', ignore_index=True)
)

In [7]:
# II. Fixed features de las vialidades
# n_vialidades, max_carriles, both_directions, via_primaria, via_acc_cont, max_nivel
fixed_features = (
    gpd
    .sjoin(
        left_df=grid,
        right_df=vialidades,
        how="left", 
        predicate="intersects"
    )
    .groupby("grid_id")
    .agg(
        n_vialidades=pd.NamedAgg("id_via", "nunique"),
        max_carriles=pd.NamedAgg("carriles", "max"),
        both_directions=pd.NamedAgg("sentidos", lambda x: int(2 in list(x))),
        via_primaria=pd.NamedAgg("tipo_vialidad", lambda x: int("primaria" in set(x))),
        via_acc_cont=pd.NamedAgg("tipo_vialidad", lambda x: int("acceso_controlado" in set(x))),
        max_nivel=pd.NamedAgg("nivel", "max")
    )
    .reset_index()
)

In [8]:
# III. Datos de afluencia y distancia a estaciones del metro
affluence = (
    afluencia
    .query('before_treatment') # previo a la implementación de las cámaras
    .groupby('estacion')
    .agg(
        amm=pd.NamedAgg("afluencia_total", "mean") # afluencia media mensual
    ) 
    .reset_index()
    .merge(coordinates, on='estacion')
    .pipe(lambda df: (
        gpd
        .sjoin_nearest(
            left_df=grid.set_geometry('centroid').to_crs(epsg=6933), # transformamos ambos sistemas de referencia a metros
            right_df=df.set_geometry('geometry').to_crs(epsg=6933),
            how='inner', 
            distance_col='distancia',
        )
    ))
    .drop_duplicates(subset='grid_id')
    # escalamos los datos
    # utilizo el segundo mayor como el mayor 
    # Porque Pantitlán es por muuucho más transitado, y luego lo corto a 1 como máximo
    .assign(
        amm=lambda x: ((x.amm - x.amm.min()) / (x.amm.nlargest(2).iloc[1] - x.amm.min())).clip(upper=1),
        distance_to_station=lambda x: ((x.distancia - x.distancia.min()) / (x.distancia.max() - x.distancia.min()))
    )
    [['grid_id', 'amm', 'distance_to_station']]
)

In [11]:
# IV. Accidents data
# Nota: Aquí es donde se puede hacer el análisis de los accidentes 
# filtrando solo por los parámetros deseados. 
# Eg. Solo accidentes en fines de semana, solo accidentes a ciertas horas del día, etc. 
accidents_in_grid = (
    gpd
    .sjoin(
        left_df=grid,
        right_df=accidents.query('before_treatment'), # accidentes antes de las 6 de la mañana
        how='left', 
        predicate='intersects'
    )
    .pivot_table(
        index='grid_id', 
        columns='incident_level',
        values='folio',
        aggfunc='nunique'
    )
    .rename(lambda x: x.lower(), axis=1)
    .assign(
        total_accidents=lambda x: x.sum(axis=1)
    )
    # Y finalmente escalamos todos los datos y ya está
    .assign(
        #fcs=lambda x: ((x.fcs - x.fcs.min()) / (x.fcs.max() - x.fcs.min())),
        #min=lambda x: ((x['min'] - x['min'].min()) / (x['min'].max() - x['min'].min())),
        #pic=lambda x: ((x.pic - x.pic.min()) / (x.pic.max() - x.pic.min())),
        #total_accidents=lambda x: ((x.total_accidents - x.total_accidents.min()) / (x.total_accidents.max() - x.total_accidents.min()))
    )
    .reset_index()
)

In [13]:
accidents_in_grid.sort_values("total_accidents")

incident_level,grid_id,fcs,min,pic,total_accidents
0,316,0,0,0,0
10604,119532,0,0,0,0
10607,119550,0,0,0,0
10608,119570,0,0,0,0
10612,119589,0,0,0,0
...,...,...,...,...,...
18589,189760,1,151,85,237
12708,131069,4,169,82,255
5069,87780,1,165,91,257
16740,160745,1,181,106,288


In [88]:
# V. Creamos el dataframe con el número de accidentes tras la intervención
mean_monthly_accidents_at = (
    gpd
    .sjoin(
        left_df=grid,
        right_df=accidents.query('~before_treatment'), # accidentes después del tratamiento
        how='left', 
        predicate='intersects'
    )
    .assign(
        accident=lambda x: np.where(x.timestamp.isna(), 0, 1)
    )
    # contamos accidentes mensuales
    .groupby([
        'grid_id', 
        'incident_level',
        pd.Grouper(key='timestamp', freq='m')
    ])
    .agg(
        accidents=pd.NamedAgg('accident', 'sum')
    )
    .reset_index()
    # agrupamos por mes y calculamos medias
    .pivot(index=['grid_id', 'timestamp'], columns='incident_level', values='accidents')
    .assign(TOTAL=lambda x: x.sum(axis=1))
    .reset_index()
    .groupby('grid_id')
    .agg(
        fcs=pd.NamedAgg('FCS', 'mean'),
        pic=pd.NamedAgg('PIC', 'mean'),
        min=pd.NamedAgg('MIN', 'mean'),
        total=pd.NamedAgg('TOTAL', 'mean')
    )
    .reset_index()
)

mean_monthly_accidents_at.to_parquet(os.path.join(PATH_DATA, "mean-monthly-accidents-at.parquet"), index=False)

### Juntamos todos los features

In [89]:
final_ps_features = (
    grid_has_camera
    .merge(fixed_features)
    .merge(affluence)
    .merge(accidents_in_grid)
)
final_ps_features.to_parquet(os.path.join(PATH_DATA, "grid-propensity-score-features.parquet"), index=False)